In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from collections import Counter

nltk.download('stopwords')

# Check if GPU is available (Colab gives you free GPU!)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
columns = ['target', 'id', 'date', 'flag', 'user', 'text']

df = pd.read_csv('/content/sample_data/training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=columns,
                 on_bad_lines='skip',
                 engine='python')

# Keep only what we need
df = df[['target', 'text']]

# Convert labels: 4 → 1 (positive), 0 stays 0 (negative)
df['target'] = df['target'].replace(4, 1)

# Sample 200K balanced (100K per class)
df_sample = df.groupby('target').apply(
    lambda x: x.sample(100000, random_state=42)
).reset_index(drop=True)

print(f"Dataset shape: {df_sample.shape}")
print(f"Label distribution:\n{df_sample['target'].value_counts()}")

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = text.strip()
    words = [w for w in text.split() if w not in stop_words and len(w) > 1]
    return words  # Return LIST of words (not string) for neural network

print("Cleaning tweets... ⏳")
df_sample['tokens'] = df_sample['text'].apply(clean_tweet)
print("Done!")

# Build vocabulary from training data
print("Building vocabulary...")
all_words = [word for tokens in df_sample['tokens'] for word in tokens]
word_counts = Counter(all_words)

# Keep top 20,000 most common words
VOCAB_SIZE = 20000
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(VOCAB_SIZE - 2)]
word2idx = {word: idx for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")
print

In [ ]:
MAX_LEN = 30  # Max words per tweet

def encode_tokens(tokens, word2idx, max_len):
    """Convert words to indices, pad or truncate to max_len"""
    indices = [word2idx.get(w, 1) for w in tokens]  # 1 = <UNK>
    # Pad with 0 (<PAD>) or truncate
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

class TweetDataset(Dataset):
    def __init__(self, df, word2idx, max_len):
        self.labels = torch.tensor(df['target'].values, dtype=torch.float32)
        self.texts = torch.tensor(
            [encode_tokens(tokens, word2idx, max_len) for tokens in df['tokens']],
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Split 80/20 train/test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_sample, test_size=0.2, random_state=42, stratify=df_sample['target'])

# Create datasets
train_dataset = TweetDataset(train_df.reset_index(drop=True), word2idx, MAX_LEN)
test_dataset  = TweetDataset(test_df.reset_index(drop=True), word2idx, MAX_LEN)

# Create dataloaders
BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples:   {len(train_dataset)}")
print(f"Testing samples:    {len(test_dataset)}")
print(f"Batches per epoch:  {len(train_loader)}")
print(f"\nSample batch shapes:")
texts, labels = next(iter(train_loader))
print(f"  Text tensor:  {texts.shape}")
print(f"  Label tensor: {labels.shape}")

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super(SentimentLSTM, self).__init__()

        # Embedding layer: converts word indices → dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # LSTM layer: learns sequential patterns in text
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True  # reads text both forward AND backward
        )

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Final classifier
        # bidirectional → hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, 1)

        # Sigmoid for binary output (0 or 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded shape: (batch_size, seq_len, embed_dim)

        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Take last hidden state from both directions
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        # hidden shape: (batch_size, hidden_dim * 2)

        out = self.dropout(hidden)
        out = self.fc(out)
        out = self.sigmoid(out)
        return out.squeeze()

# Hyperparameters
VOCAB_SIZE  = len(vocab)
EMBED_DIM   = 128
HIDDEN_DIM  = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3

model = SentimentLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model architecture:")
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

In [ ]:
EPOCHS = 5
LEARNING_RATE = 0.001

criterion = nn.BCELoss()  # Binary Cross Entropy for binary classification
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        predicted = (predictions >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            predictions = model(texts)
            loss = criterion(predictions, labels)
            total_loss += loss.item()
            predicted = (predictions >= 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

print("Starting training...")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Train Acc':<14}{'Val Loss':<14}{'Val Acc':<12}")
print("-" * 60)

best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = evaluate(model, test_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    marker = ' ⭐' if val_acc > best_val_acc else ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_lstm_model.pt')

    print(f"{epoch:<8}{train_loss:<14.4f}{train_acc:<14.4f}{val_loss:<14.4f}{val_acc:<12.4f}{marker}")

print(f"\nTraining complete! Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='#e74c3c', marker='o')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#38BDF8', marker='o')
axes[0].set_title('Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', color='#e74c3c', marker='o')
axes[1].plot(history['val_acc'],   label='Val Acc',   color='#38BDF8', marker='o')
axes[1].set_title('Accuracy over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Bidirectional LSTM — Sentiment Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Load best model and evaluate
model.load_state_dict(torch.load('best_lstm_model.pt'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        preds = model(texts)
        predicted = (preds >= 0.5).float().cpu().numpy()
        all_preds.extend(predicted)
        all_labels.extend(labels.numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds,
                            target_names=['Negative', 'Positive']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix — Bidirectional LSTM', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def predict_sentiment(tweet, model, word2idx, max_len=30):
    model.eval()
    tokens = clean_tweet(tweet)
    encoded = encode_tokens(tokens, word2idx, max_len)
    tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    with torch.no_grad():
        prob = model(tensor).item()
    sentiment = 'Positive' if prob >= 0.5 else 'Negative'
    confidence = prob if prob >= 0.5 else 1 - prob
    return sentiment, confidence

test_tweets = [
    "I absolutely love this product, it changed my life!",
    "This is the worst experience I have ever had",
    "Just had an amazing day with my friends!",
    "I hate waiting in long queues, so frustrating",
    "The weather today is perfect for a walk",
    "Feeling so tired and exhausted today"
]

print("=" * 58)
print("     BIDIRECTIONAL LSTM — SENTIMENT PREDICTIONS")
print("=" * 58)

for tweet in test_tweets:
    sentiment, confidence = predict_sentiment(tweet, model, word2idx)
    print(f"\nTweet:      {tweet}")
    print(f"Sentiment:  {sentiment}")
    print(f"Confidence: {confidence*100:.1f}%")
    print("-" * 58)